# 🤝 Agent Handoffs in LangGraph

## Learning Objectives
In this notebook, you will learn:
1. **Router vs handoff** - Why a supervisor that routes and takes control back is not a handoff, and what changes when an agent passes the conversation on itself
2. **The handoff tool** - Build a tool that returns `Command(goto=…, graph=Command.PARENT, update={…})`, the mechanism the `langgraph-swarm` library uses
3. **Triage and specialists** - Wire `create_agent` agents into a parent `StateGraph`: triage answers simple questions itself or hands off with a context note, and specialists hand off sideways or back to triage
4. **Who is active** - Persist `active_agent` with a checkpointer, so the next message on a thread goes straight to the agent that holds the conversation
5. **Loop guard** - Stop agents that keep handing off to each other with `recursion_limit`

## Where this fits
Module 06's `02_hierarchical_agents.ipynb` keeps a CEO in charge: it routes on a typed decision and takes control
back after every department. A **handoff** is different: the agent that holds the conversation decides to pass it
on, by calling a tool, and steps out. This notebook builds handoffs by hand; `02_multi_agent_swarm.ipynb` uses the
`langgraph-swarm` library, which packages the same mechanism.

## Prerequisites
- `02_hierarchical_agents.ipynb` (module 06); `Command(goto=…)` from `03_five_multi_agent_architectures.ipynb` (module 05); checkpointers and `thread_id` (module 02)
- Agents built with `langchain.agents.create_agent` and tools built with `@tool`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env` (Part 2's tool call and the scripted runs in 4.1 and Part 5 make no API call)

---
## 📦 Step 1: Environment Setup

The setup cell imports everything the notebook uses, loads the API keys from the project-root `.env`, silences library deprecation warnings and sets `RECURSION_LIMIT`, the cap every run passes (Part 5).

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys, imports, and the loop cap
# ============================================================================
import itertools
import uuid
import warnings
from typing import Annotated, Any

from dotenv import find_dotenv, load_dotenv
from langchain.agents import create_agent
from langchain.tools import InjectedState, InjectedToolCallId, tool
from langchain_core.language_models import GenericFakeChatModel
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.errors import GraphRecursionError
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.types import Command

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

# Every run passes this cap. LangGraph counts super-steps, and its default
# (10,007) lets a handoff loop spin for a long time before failing (Part 5).
RECURSION_LIMIT = 25

print(f"✅ Environment ready (recursion_limit = {RECURSION_LIMIT})")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: triage and the three specialists
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🔀 Part 1: Router vs Handoff

Both patterns move a request between agents. They differ in **who holds control afterwards**.

In `02_hierarchical_agents.ipynb` (module 06) the CEO is a **router**. It returns a typed `DepartmentRoute`, a pure
function routes on it, and every department edge leads back to `ceo`. The CEO decides every step and stays in charge
until it chooses `FINISH`; a department never picks the next department.

In a **handoff**, the agent that holds the conversation decides to pass it on. It calls a tool such as
`transfer_to_billing`. The tool tells the parent graph to run `billing` next and records `billing` as the
**active agent**. Billing now owns the conversation: no edge leads back to a coordinator, and the customer's next
message goes straight to billing.

| | Router (the 06/02 CEO) | Handoff (this notebook) |
|---|---|---|
| Who picks the next agent | the router, after every step | the agent holding the conversation, by calling a tool |
| Where control goes after an agent | back to the router (a static edge) | nowhere: the agent finishes the turn or hands off |
| The next user message starts at | the router | the active agent (`active_agent` in state) |
| What stops a loop | the router's own cap | nothing built in: `recursion_limit` (Part 5) |

> **Key Insight**: a structured-output classifier that sends each request to a specialist once is still a router.
> A handoff needs three things: the agent itself chooses (a tool call), the parent graph moves control
> (`Command.PARENT`), and the choice is remembered (`active_agent` plus a checkpointer).

---
## 🛠️ Part 2: The Handoff Tool

A handoff tool is an ordinary `@tool` whose return value is a `Command` instead of a string. The tool runs inside an
agent's own graph (its `ToolNode`), but the decision belongs to the **parent** graph that holds all the agents, so the
`Command` is addressed to the parent:

- **`goto=agent_name`** - the parent node to run next
- **`graph=Command.PARENT`** - apply this `Command` in the closest parent graph, not inside the calling agent
- **`update={...}`** - what to write to the parent state: the messages so far plus a `ToolMessage` that answers the tool call, and **`active_agent`**

This mirrors `create_handoff_tool` in the installed `langgraph_swarm/handoff.py`, which `02_multi_agent_swarm.ipynb`
uses. One difference: our tool takes a `context` argument, the note the next agent reads so the customer is not asked
again. The library's tool takes no arguments, so context reaches the next agent only through the messages.

### Key Concepts:
- **`InjectedState`**: the tool receives the calling agent's state (its `messages`) without the model passing it
- **`InjectedToolCallId`**: the tool receives the id of the call it answers, so its `ToolMessage` closes that call
- **`HandoffState`**: the parent's state, `MessagesState` plus `active_agent`, the same shape as `langgraph_swarm.SwarmState`. A `Command.PARENT` update may only use keys the parent state has.

In [ ]:
# ============================================================================
# HANDOFF TOOL: a @tool that returns Command(goto=..., graph=Command.PARENT)
# ============================================================================
class HandoffState(MessagesState):
    """Parent graph state: the conversation plus the agent that holds it."""

    active_agent: str | None  # written by every handoff tool; START routes on it


def make_handoff_tool(agent_name: str, description: str):
    """Build `transfer_to_<agent_name>` (mirrors langgraph_swarm.create_handoff_tool)."""
    tool_name = f"transfer_to_{agent_name}"

    @tool(tool_name, description=description)
    def handoff(
        context: str,  # the only argument the model fills in: the handoff note
        state: Annotated[Any, InjectedState],  # the calling agent's state, injected
        tool_call_id: Annotated[str, InjectedToolCallId],  # the id of this tool call, injected
    ) -> Command:
        # Every tool call must be answered by a ToolMessage with its id; this one carries the note
        note = ToolMessage(
            content=f"Transferred to {agent_name}. Handoff note: {context}",
            name=tool_name,
            tool_call_id=tool_call_id,
        )
        return Command(
            goto=agent_name,  # the parent node to run next
            graph=Command.PARENT,  # apply this in the parent graph, not inside the calling agent
            update={
                "messages": [*state["messages"], note],  # the conversation so far + the note
                "active_agent": agent_name,  # who answers the next message on this thread
            },
        )

    return handoff


# One tool per destination; the description tells the model when to use it and what to put in `context`
WHEN = {
    "sales": "Transfer the customer to sales: products, plans, purchases, upgrades.",
    "support": "Transfer the customer to technical support: crashes, bugs, errors, how-to questions.",
    "billing": "Transfer the customer to billing: payments, charges, invoices, refunds.",
    "triage": "Hand the customer back to triage: anything outside your speciality, such as opening hours.",
}
NOTE = (" Put everything the next agent needs in `context` (the customer's details and request),"
        " so the customer is not asked again.")
handoff_tools = {name: make_handoff_tool(name, when + NOTE) for name, when in WHEN.items()}

print("✅ Handoff tools:", [t.name for t in handoff_tools.values()])
print("   Arguments the model sees:", list(handoff_tools["billing"].tool_call_schema.model_json_schema()["properties"]))

### 2.1 🔍 Look Inside: Call a Handoff Tool Directly

No model is needed to see what a handoff tool does. Invoke it with a full tool call (the shape a model produces), and
pass a `state` to stand in for the injected one. It returns the `Command` the parent graph will apply.

In [ ]:
# ============================================================================
# DIRECT CALL: the Command a handoff tool returns (no LLM)
# ============================================================================
customer = HumanMessage(content="I was charged twice for my subscription this month.")
tool_call = {
    "type": "tool_call",  # a full tool call, so InjectedToolCallId receives "call_demo"
    "id": "call_demo",
    "name": "transfer_to_billing",
    "args": {
        "context": "Customer was charged twice for this month's subscription.",
        "state": {"messages": [customer]},  # stands in for the injected agent state
    },
}
command = handoff_tools["billing"].invoke(tool_call)
note = command.update["messages"][-1]

print(f"type:         {type(command).__name__}")
print(f"goto:         {command.goto}")
print(f"graph:        {command.graph}")
print(f"update keys:  {list(command.update)}")
print(f"active_agent: {command.update['active_agent']}")
print(f"note:         {note.content}  (answers {note.tool_call_id})")

ok = (
    command.goto == "billing"
    and command.graph == Command.PARENT
    and command.update["active_agent"] == "billing"
    and isinstance(note, ToolMessage)
    and note.tool_call_id == "call_demo"
)
print("✅ the tool returned a Command for the parent graph: run billing, make billing active, answer the call"
      if ok else "❌ unexpected Command")

`graph` prints `__parent__`: the `Command` ends the calling agent's run and is applied one level up. `update` carries
the whole conversation plus the new `ToolMessage`. The parent's `add_messages` reducer merges messages by id, so the
messages it already holds are not added twice. The model's only job, later, is to pick the tool and write `context`.

---
## 🏗️ Part 3: Triage, Specialists and the Parent Graph

Four agents, each built with `create_agent` (`system_prompt=` sets its role) and added to a parent
`StateGraph(HandoffState)` as one node:

| Agent | Can hand off to | Why |
|---|---|---|
| `triage` | `sales`, `support`, `billing` | the first contact; answers simple questions (opening hours) itself |
| `sales` | `triage` | hand-back for anything outside sales |
| `support` | `billing`, `triage` | a **sideways** handoff when a technical issue turns into a billing one |
| `billing` | `triage` | hand-back for anything outside billing |

The parent graph makes **one routing decision of its own**: at `START` it reads `active_agent` and sends the new
message to that agent (to `triage` on a new thread). After that, control moves only through handoff tools. An agent
that answers without handing off simply finishes: its node has no outgoing edge, so the run ends.

Only triage knows the opening hours, so a specialist must hand an opening-hours question back to triage.

In [ ]:
# ============================================================================
# AGENT ROLES: system prompts and who may hand off to whom
# ============================================================================
TRIAGE_PROMPT = """You are the triage agent of a software company's customer service desk.
Answer simple general questions yourself, using only these facts:
- Opening hours: Monday to Friday 9:00-18:00, Saturday 10:00-14:00, closed on Sunday.
- Support email: help@example.com
Hand every other request to exactly ONE specialist with a transfer tool, and do not answer it yourself:
- sales: products, plans, purchases, upgrades
- support: crashes, bugs, errors, how-to questions
- billing: payments, charges, invoices, refunds"""

SALES_PROMPT = """You are the sales specialist. Help with products, plans and upgrades; be helpful, not pushy.
Read any handoff note in the conversation first and do not ask again for details it gives.
For anything that is not about products, plans or purchases, call transfer_to_triage. Never answer it yourself."""

SUPPORT_PROMPT = """You are the technical support specialist. Be patient and give step-by-step guidance.
Read any handoff note in the conversation first and do not ask again for details it gives.
If the customer asks about payments, charges, invoices or refunds, call transfer_to_billing.
For anything else that is not a technical issue, call transfer_to_triage. Never answer those yourself."""

BILLING_PROMPT = """You are the billing specialist. Be clear about policies and next steps:
duplicate charges are refunded within 5-7 business days.
Read any handoff note in the conversation first and do not ask again for details it gives.
For anything that is not about payments, charges, invoices or refunds, call transfer_to_triage.
Never answer it yourself."""

AGENTS = {  # name: (system prompt, agents it may hand off to)
    "triage": (TRIAGE_PROMPT, ["sales", "support", "billing"]),
    "sales": (SALES_PROMPT, ["triage"]),
    "support": (SUPPORT_PROMPT, ["billing", "triage"]),
    "billing": (BILLING_PROMPT, ["triage"]),
}
print("✅ Handoff map:", {name: targets for name, (_, targets) in AGENTS.items()})

In [ ]:
# ============================================================================
# BUILD_HANDOFF_SYSTEM: four create_agent agents as nodes of one parent graph
# ============================================================================
def route_to_active_agent(state: HandoffState) -> str:
    """START: send the new message to the agent that holds the conversation (triage on a new thread)."""
    return state.get("active_agent") or "triage"


def build_handoff_system(model, checkpointer=None):
    """Compile the parent graph. `model` is an argument so a scripted stand-in can replace the LLM (4.1, Part 5)."""
    builder = StateGraph(HandoffState)
    for name, (prompt, targets) in AGENTS.items():
        agent = create_agent(
            model,
            tools=[handoff_tools[target] for target in targets],
            system_prompt=prompt,
            name=name,  # also stamped on every AIMessage this agent writes
        )
        # `destinations` only labels the possible handoffs in the graph drawing;
        # the Command's goto does the routing at run time
        builder.add_node(name, agent, destinations=tuple(targets))
    builder.add_conditional_edges(START, route_to_active_agent, list(AGENTS))
    return builder.compile(checkpointer=checkpointer)


handoff_app = build_handoff_system(llm)

graph = handoff_app.get_graph()
print("Nodes:", [n for n in graph.nodes if not n.startswith("__")])
for edge in graph.edges:
    print(f"  {edge.source} -> {edge.target}{'  (conditional)' if edge.conditional else ''}")

### 3.1 ▶️ Four Customers, Four Routes

Each customer gets a new thread, so every run begins at triage. `run_turn` sends one message and reads the result
from the messages this turn added:
- **agents this turn** - the `name` stamped on each `AIMessage`, in order (`triage → billing` means triage handed off to billing)
- **🔀 lines** - the `ToolMessage` each handoff tool wrote, with the forwarded note
- **active_agent** - the parent state after the turn

The crash should go to **support**, the upgrade to **sales**, the double charge to **billing**, and triage should answer
the opening-hours question itself. The billing customer gives an account email; a second ✅ checks that triage
forwarded it in its handoff note.

In [ ]:
# ============================================================================
# RUN_TURN: send one customer message and show who handled it
# ============================================================================
def run_turn(app, thread_id: str, text: str):
    """Send one message on a thread; print and return (agents that acted, handoff notes, final state)."""
    config = {"configurable": {"thread_id": thread_id}, "recursion_limit": RECURSION_LIMIT}
    result = app.invoke({"messages": [HumanMessage(content=text)]}, config)

    # Only the messages this turn added: everything after the last customer message
    last_human = max(i for i, m in enumerate(result["messages"]) if isinstance(m, HumanMessage))
    turn = result["messages"][last_human + 1 :]

    agents = []  # agent names in the order they acted, consecutive repeats collapsed
    for message in turn:
        if isinstance(message, AIMessage) and message.name and (not agents or agents[-1] != message.name):
            agents.append(message.name)
    notes = [m.content for m in turn if isinstance(m, ToolMessage)]

    print(f"Customer: {text}")
    print(f"  agents this turn: {' → '.join(agents)}")
    for line in notes:
        print(f"  🔀 {line}")
    print(f"  reply: {turn[-1].content[:250]}")
    print(f"  active_agent: {result.get('active_agent')}")
    return agents, notes, result


print("✅ run_turn() defined")

In [ ]:
# ============================================================================
# RUN: FOUR CUSTOMERS, FOUR ROUTES (a new thread each)
# ============================================================================
CASES = [  # (customer message, expected agents this turn)
    ("My app keeps crashing when I try to upload photos.", ["triage", "support"]),
    ("I want to upgrade to the premium plan.", ["triage", "sales"]),
    ("I was charged twice for my subscription this month. My account is jane@example.com.", ["triage", "billing"]),
    ("What time do you close on Saturdays?", ["triage"]),
]

passed = 0
for text, expected in CASES:
    agents, notes, _ = run_turn(handoff_app, str(uuid.uuid4()), text)
    ok = agents == expected
    passed += ok
    print(f"  {'✅' if ok else '❌'} expected {' → '.join(expected)}")
    if "jane@example.com" in text:
        forwarded = any("jane@example.com" in line for line in notes)
        print(f"  {'✅' if forwarded else '❌'} triage forwarded the account email in its handoff note")
    print("-" * 60)

print(f"{passed}/{len(CASES)} customers reached the expected agent")

Read each block. A handoff shows two agents and a 🔀 line, and the specialist's reply should build on the note
instead of asking again (billing should not ask for the account email). The opening-hours customer shows only
`triage` and no 🔀 line: triage answered from its own prompt, and `active_agent` stays `None` because no handoff
happened. `handoff_app` has no checkpointer yet, so `active_agent` is lost as soon as each run returns.

---
## 💾 Part 4: Remember Who Is Active

A handoff is only half done if the next message goes back to triage. Compiling the same graph with a checkpointer
(`InMemorySaver`) saves the parent state, messages **and** `active_agent`, per `thread_id`. On the next message the
`START` router reads the saved `active_agent` and sends the message straight to that agent.

One thread, four turns:
1. A crash report → triage hands off to **support** (`active_agent` = support)
2. A follow-up with device details → starts at **support**; triage does not run
3. A refund request → support hands off **sideways** to **billing**
4. An opening-hours question → billing **hands back** to **triage**, which answers

In [ ]:
# ============================================================================
# ONE THREAD, FOUR TURNS: active_agent saved by the checkpointer
# ============================================================================
memory_app = build_handoff_system(llm, checkpointer=InMemorySaver())
thread_id = str(uuid.uuid4())

SCRIPT = [  # (customer message, expected agents this turn, what the turn shows)
    ("My app crashes every time I upload a photo. My account is jane@example.com.",
     ["triage", "support"], "triage hands off to support"),
    ("I'm on an iPhone 15 with iOS 18. What should I try first?",
     ["support"], "turn 2 starts at support, not triage"),
    ("That fixed it, thanks! But I was charged twice for premium this month. Can I get a refund?",
     ["support", "billing"], "sideways handoff: support → billing"),
    ("One more thing: what time do you close on Saturdays?",
     ["billing", "triage"], "hand-back: billing → triage"),
]

passed = 0
for n, (text, expected, shows) in enumerate(SCRIPT, 1):
    print(f"--- Turn {n}: {shows}")
    agents, _, _ = run_turn(memory_app, thread_id, text)
    saved = memory_app.get_state({"configurable": {"thread_id": thread_id}}).values.get("active_agent")
    ok = agents == expected and saved == expected[-1]
    passed += ok
    print(f"  saved in the checkpoint: active_agent = {saved}")
    print(f"  {'✅' if ok else '❌'} expected {' → '.join(expected)}, then active_agent = {expected[-1]}\n")

print(f"{passed}/{len(SCRIPT)} turns took the expected path")

Turn 2 is the point of the checkpointer: the first agent to act is support, because `START` read
`active_agent = support` from the saved state. On a new thread nothing is saved, so the router falls back to triage.
Turns 3 and 4 show that control also moves sideways and back. Every move is one handoff tool call, and each one
rewrites `active_agent`.

### 4.1 🧪 The Same Mechanism, Proved Without an LLM

The routes above depend on the model's choices. The mechanism does not: replace the LLM with a **scripted
stand-in** that replays fixed replies, and the same graph shows the same behaviour with no API call.

`ScriptedModel` is LangChain's `GenericFakeChatModel` (it returns the next message from a list) with `bind_tools`
returning the model itself: `create_agent` binds the tools, and the scripted replies already contain their tool
calls. The script: triage calls `transfer_to_billing`, billing answers, and on turn 2 billing answers again.

In [ ]:
# ============================================================================
# SCRIPTED PROOF: turn 2 starts at the active agent (no LLM)
# ============================================================================
class ScriptedModel(GenericFakeChatModel):
    """A stand-in chat model: replays scripted replies, makes no API call."""

    def bind_tools(self, tools, **kwargs):
        return self  # the scripted replies already contain their tool calls


def transfer_call(target: str, context: str, call_id: str) -> AIMessage:
    """A scripted model reply that calls transfer_to_<target>."""
    return AIMessage(
        content="", tool_calls=[{"name": f"transfer_to_{target}", "args": {"context": context}, "id": call_id}]
    )


script = iter([
    transfer_call("billing", "Customer was charged twice this month.", "call_1"),  # turn 1: triage
    AIMessage(content="Refund started; it reaches you within 5-7 business days."),  # turn 1: billing
    AIMessage(content="Yes, you get a confirmation email once it is processed."),  # turn 2: billing
])
scripted_app = build_handoff_system(ScriptedModel(messages=script), checkpointer=InMemorySaver())
scripted_thread = str(uuid.uuid4())

turn1, _, _ = run_turn(scripted_app, scripted_thread, "I was charged twice.")
turn2, _, state = run_turn(scripted_app, scripted_thread, "Will I get a confirmation?")

ok = turn1 == ["triage", "billing"] and turn2 == ["billing"] and state["active_agent"] == "billing"
print("✅ turn 1 handed off to billing; turn 2 started at billing, not triage (no LLM call)"
      if ok else "❌ unexpected path")

---
## 🛡️ Part 5: Guard Against Handoff Loops

A handoff system has no router that can say "stop". If two agents each decide the other should handle a request,
they pass it back and forth forever, and LangGraph's default `recursion_limit` is 10,007 super-steps. Every run in
this notebook passes `recursion_limit=RECURSION_LIMIT` in its config; each agent run is one super-step of the parent
graph.

To see the guard work without paying for a loop, the scripted model below makes every agent hand off at once:
triage → support → billing → triage → … The run must end with `GraphRecursionError`, caught in code.
`03_five_multi_agent_architectures.ipynb` (module 05) shows the same guard on two plain nodes.

In [ ]:
# ============================================================================
# LOOP GUARD: endless handoffs stopped by recursion_limit (no LLM)
# ============================================================================
LOOP_LIMIT = 10  # a small cap for this demo; the real runs above use RECURSION_LIMIT
handoffs = []  # one entry per scripted handoff


def endless_handoffs():
    """Scripted replies: triage -> support -> billing -> triage -> ... forever."""
    for target in itertools.cycle(["support", "billing", "triage"]):
        handoffs.append(target)
        yield transfer_call(target, "Not mine, passing it on.", f"call_{len(handoffs)}")


looping_app = build_handoff_system(ScriptedModel(messages=endless_handoffs()))
try:
    looping_app.invoke({"messages": [HumanMessage(content="Help!")]}, {"recursion_limit": LOOP_LIMIT})
    print("❌ the loop ended on its own")
except GraphRecursionError:
    print(f"Handoffs before the cap: {len(handoffs)}  (triage → {' → '.join(handoffs[:6])} → …)")
    print(f"✅ recursion_limit={LOOP_LIMIT} stopped the handoff loop with GraphRecursionError")

---
## 📝 Summary

In this notebook, we learned:

### 1. Router vs handoff
- A router decides every step and takes control back (the 06/02 CEO). A handoff lets the agent holding the conversation pass it on, and the receiver keeps it.

### 2. The handoff tool
- A `@tool` that returns `Command(goto=agent, graph=Command.PARENT, update={"messages": …, "active_agent": agent})`, the same shape as `langgraph_swarm.create_handoff_tool`.
- `InjectedState` and `InjectedToolCallId` give the tool the conversation and its call id; the model fills in only `context`, the handoff note.

### 3. Triage and specialists
- Each agent is a `create_agent` agent and one node of a parent `StateGraph(HandoffState)`; the only routing the parent does itself is `START → active_agent`.
- Triage answers simple questions itself and forwards a note when it hands off; specialists hand off sideways (`support → billing`) or back to triage.

### 4. Who is active
- With a checkpointer, `active_agent` outlives the turn: the next message on the thread starts at the agent that holds the conversation.
- A scripted model proves this without an LLM. That works because `build_handoff_system` takes the model as an argument.

### 5. Loop guard
- Nothing in a handoff system forces it to stop. `recursion_limit` in the run config turns an endless handoff loop into a `GraphRecursionError`.

### Next Steps
- **`02_multi_agent_swarm.ipynb`**: the same mechanism from the `langgraph-swarm` library (`create_handoff_tool`, `create_swarm`), with RAG, SQL and invoice agents